In [ ]:
from pathlib import Path

import pandas as pd

from chef_classifier.data import (
    clean_training_data,
    create_train_val_split,
    load_training_data,
)

In [ ]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test-no-labels.csv"

In [ ]:
train = load_training_data(TRAIN_PATH)
test = pd.read_csv(TEST_PATH, sep=";")

In [ ]:
train.head()

In [ ]:
train.shape, test.shape

In [ ]:
train.columns.tolist(), test.columns.tolist()

In [ ]:
train.info()

In [ ]:
train["chef_id"].value_counts()

In [ ]:
train["chef_id"].value_counts(normalize=True)

In [ ]:
train["chef_id"].value_counts().sort_index().plot(
    kind="bar",
    title="Number of recipes per chef",
    xlabel="Chef ID",
    ylabel="Number of recipes",
)

In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
text_columns = [
    "recipe_name",
    "tags",
    "steps",
    "description",
    "ingredients",
]

for col in text_columns:
    print(
        col,
        "avg chars:",
        train[col].str.len().mean(),
        "median chars:",
        train[col].str.len().median(),
    )

In [ ]:
for col in [
    "recipe_name",
    "description",
    "ingredients",
    "steps",
    "tags",
]:
    train[f"{col}_length"] = train[col].str.len()

train.groupby("chef_id")[
    [
        "recipe_name_length",
        "description_length",
        "ingredients_length",
        "steps_length",
        "tags_length",
    ]
].mean().round(1)

In [ ]:
train.groupby("chef_id")["n_ingredients"].describe()

In [ ]:
train.nunique()

In [ ]:
train.duplicated().sum()

In [ ]:
for col in [
    "recipe_name",
    "description",
    "ingredients",
    "steps",
    "tags",
]:
    print(col, train[col].duplicated().sum())

In [ ]:
duplicate_names = train[
    train["recipe_name"].duplicated(keep=False)
].sort_values("recipe_name")

duplicate_names[
    ["chef_id", "recipe_name", "description", "ingredients"]
]

In [ ]:
train["parsed_date"] = pd.to_datetime(
    train["data"],
    format="%d/%m/%Y",
)

train.groupby("chef_id")["parsed_date"].agg(["min", "max"])

In [ ]:
train["year"] = train["parsed_date"].dt.year

pd.crosstab(
    train["year"],
    train["chef_id"],
)

In [ ]:
train.groupby("chef_id")["year"].describe()

In [ ]:
for col in ["recipe_name", "description", "ingredients", "tags"]:
    print(f"\n--- {col} ---")
    for value in train[col].sample(3, random_state=42):
        print(value[:500])
        print()

In [ ]:
train_clean = clean_training_data(train)
train_clean.shape

In [ ]:
train_df, val_df = create_train_val_split(train_clean)

In [ ]:
train_df["chef_id"].value_counts(normalize=True).sort_index()

In [ ]:
val_df["chef_id"].value_counts(normalize=True).sort_index()

- The training set has 2999 recipes, the test set has 823 recipes.
- The training set has 8 columns, the test set has those same features except for the target variable chef_id (which contains 6 chef classes).
- The class distribution is moderately imbalanced (chef 4470 is the most frequent class with 26.9% of the training examples, chef 6357 is the least frequent with 12.4%)
- There are no missing values.

- Text features: recipe_name; tags; steps; description; ingredients.
- Numeric feature: n_ingredients. (This feature appears to vary a bit between chefs)
- Temporal feature: data

- There are 14 exact duplicate rows in the training set (we should remove them). Other text fields are also almost completely unique with a few exceptions: description has 40 duplicated values; tags has 23; steps has 20; ingredients has 14.

- The date distribution differ across chefs, which suggests that date-related information may have predictive value. However, date may reflect dataset collection patterns rather than genuine linguistic or culinary style, so it should be treated as a potentially dataset-specific feature and evaluated separately.

- The average number of ingredients and text lengths also varies somewhat between chefs, suggesting that stylistic features beyond vocabulary may contain useful information.



- For validation: exact duplicated were removed; the remaining data was split into 80% training and 20% validation; this split was stratified by chef_id, ensuring more or less the same class distribution in both subsets.